# Classification morphologique de galaxies par CNN
### Galaxy10 DECaLS — 17 736 images, 10 classes

Patrick Yvan Eyike · Philémon Vittet · Sidi Hamadi · Abdelrahman Dahy · Inès Lylia Zerdani

**Question.** Un réseau de neurones convolutif peut-il reproduire le travail de tri
morphologique fait par les volontaires de Galaxy Zoo ?

On raconte une histoire en trois niveaux pour isoler l'apport de la **convolution** :

1. **baselines sans convolution** (régression logistique, MLP sur pixels aplatis) ;
2. **CNN maison** (partage de poids, équivariance par translation) ;
3. **transfer learning** (ResNet18 pré-entraîné sur ImageNet).

Puis on **interprète** avec Grad-CAM (où le réseau regarde) et on quantifie l'apport de
chaque ingrédient par **ablations**. Le lien physique est direct : l'équivariance par
translation du CNN est une symétrie imposée à l'architecture, dans l'esprit des symétries
en physique.

Sections : 0 Setup · 1 EDA · 2 Preprocessing & augmentation · 3 Baselines · 4 CNN maison &
tuning · 5 Transfer learning · 6 Évaluation test · 7 Grad-CAM · 8 Ablations.

## 0. Setup — environnement et bibliothèque

Sur Colab, activer le GPU (Exécution → Modifier le type d'exécution → T4) et lancer la
cellule d'installation. Les cellules suivantes définissent toute la bibliothèque du projet
(données, modèles, entraînement, métriques, figures, Grad-CAM) : le notebook est
auto-portant.

In [ ]:
# Colab : décommenter la ligne suivante (torch/torchvision/sklearn sont déjà présents)
# !pip -q install "datasets>=3.0.0"
import os, json, time, copy, random
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support)
from IPython.display import Image as ShowImage, display
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| mps", torch.backends.mps.is_available())

### Constantes, device et chemins

In [ ]:
"""Constantes, device et chemins du projet Galaxy10."""

import os
from pathlib import Path

import torch

# --- Classes Galaxy10 DECaLS (ordre = label 0..9) ---------------------------
CLASS_NAMES = [
    "Disturbed",
    "Merging",
    "Round Smooth",
    "In-between Round Smooth",
    "Cigar Shaped Smooth",
    "Barred Spiral",
    "Unbarred Tight Spiral",
    "Unbarred Loose Spiral",
    "Edge-on without Bulge",
    "Edge-on with Bulge",
]
# Effectifs officiels (astroNN), utiles pour l'EDA et un contrôle de cohérence.
CLASS_COUNTS = [1081, 1853, 2645, 2027, 334, 2043, 1829, 2628, 1423, 1873]
NUM_CLASSES = len(CLASS_NAMES)

HF_DATASET = "matthieulel/galaxy10_decals"
SEED = 42

# Stats ImageNet (pour le backbone ResNet pré-entraîné).
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# --- Chemins ----------------------------------------------------------------
IN_COLAB = "google.colab" in os.sys.modules or os.environ.get("COLAB_RELEASE_TAG") is not None


def project_root() -> Path:
    """Racine du projet, robuste en local comme sur Colab."""
    if IN_COLAB:
        p = Path("/content/galaxy-cnn")
        p.mkdir(parents=True, exist_ok=True)
        return p
    try:
        return Path(__file__).resolve().parent.parent
    except NameError:
        # Exécuté hors module (cellules de notebook inlinées) : on prend le cwd.
        return Path.cwd()


def data_dir() -> Path:
    d = project_root() / "data"
    d.mkdir(parents=True, exist_ok=True)
    return d


def figures_dir() -> Path:
    d = project_root() / "figures"
    d.mkdir(parents=True, exist_ok=True)
    return d


def checkpoints_dir() -> Path:
    d = project_root() / "checkpoints"
    d.mkdir(parents=True, exist_ok=True)
    return d


# --- Device -----------------------------------------------------------------
def get_device() -> torch.device:
    """CUDA (Colab) > MPS (Mac) > CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def set_seed(seed: int = SEED) -> None:
    import random

    import numpy as np

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

### Réglages d'exécution

`FULL_RUN=True` : run complet (Colab, tout le dataset, ~30 epochs). Pour itérer vite en
local, exporter `GALAXY_FULL=0` : le notebook bascule sur le petit split test (toutes les
classes) et peu d'epochs, ce qui valide toute la chaîne en quelques minutes.

In [ ]:
FULL_RUN = os.environ.get("GALAXY_FULL", "1") == "1"
RES_SMALL, RES_LARGE = 64, 128        # 64 px pour itérer, 128 px pour les runs finaux
BATCH = 128
NUM_WORKERS = 2 if torch.cuda.is_available() else 0
if FULL_RUN:
    SMOKE_N = None
    EPOCHS_BASE, EPOCHS_CNN, TUNE_EPOCHS = 15, 30, 10
    EPOCHS_RES_FE, EPOCHS_RES = 6, 12
else:
    SMOKE_N = 800
    EPOCHS_BASE, EPOCHS_CNN, TUNE_EPOCHS = 3, 4, 2
    EPOCHS_RES_FE, EPOCHS_RES = 2, 2
print("FULL_RUN =", FULL_RUN, "| SMOKE_N =", SMOKE_N)

### Métriques

In [ ]:
"""Métriques d'évaluation (numériques). Les figures sont dans viz.py."""

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)



def summary_metrics(y_true, y_pred) -> dict:
    """Accuracy + precision/recall/F1 en macro et pondéré."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    p_w, r_w, f_w, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f_macro,
        "precision_weighted": p_w,
        "recall_weighted": r_w,
        "f1_weighted": f_w,
    }


def per_class_df(y_true, y_pred, class_names=CLASS_NAMES) -> pd.DataFrame:
    """Precision/recall/F1/support par classe."""
    p, r, f, s = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(class_names)), zero_division=0
    )
    return pd.DataFrame(
        {"precision": p, "recall": r, "f1": f, "support": s}, index=class_names
    )


def confusion(y_true, y_pred, normalize: str | None = None) -> np.ndarray:
    """Matrice de confusion. normalize='true' pour des lignes en proportion."""
    n = len(CLASS_NAMES)
    return confusion_matrix(y_true, y_pred, labels=range(n), normalize=normalize)


def macro_f1(y_true, y_pred) -> float:
    return float(f1_score(y_true, y_pred, average="macro", zero_division=0))

### Modèles — baselines, CNN maison, ResNet18

In [ ]:
"""Les modèles du projet : baselines sans convolution, CNN maison, ResNet18.

L'histoire du projet est l'échelle de trois niveaux :
  1. baselines (régression logistique, MLP) aveugles à la structure spatiale ;
  2. CNN maison (convolution = partage de poids + équivariance par translation) ;
  3. transfer learning (ResNet18 pré-entraîné ImageNet).
"""

import torch
import torch.nn as nn



class LogisticRegression(nn.Module):
    """Baseline 1 : régression logistique multinomiale sur pixels aplatis.

    Une seule couche linéaire. Aucune notion de localité : chaque pixel est une
    variable indépendante. Volontairement médiocre.
    """

    def __init__(self, in_channels: int = 3, image_size: int = 64, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.in_features = in_channels * image_size * image_size
        self.fc = nn.Linear(self.in_features, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x.flatten(1))


class MLP(nn.Module):
    """Baseline 2 : perceptron multicouche sur pixels aplatis.

    Deux couches cachées + ReLU + dropout. Plus expressif que la régression
    logistique, mais toujours aveugle à la structure 2D.
    """

    def __init__(
        self,
        in_channels: int = 3,
        image_size: int = 64,
        hidden=(512, 256),
        num_classes: int = NUM_CLASSES,
        dropout: float = 0.5,
    ):
        super().__init__()
        in_features = in_channels * image_size * image_size
        layers: list[nn.Module] = [nn.Flatten()]
        prev = in_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(inplace=True), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def _conv_block(in_c: int, out_c: int, use_bn: bool = True) -> nn.Sequential:
    """Bloc [Conv 3x3 pad 1 -> (BatchNorm) -> ReLU -> MaxPool 2].

    La conv 3x3 pad 1 conserve la taille spatiale ; le pooling la divise par 2.
    """
    layers: list[nn.Module] = [nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, bias=not use_bn)]
    if use_bn:
        layers.append(nn.BatchNorm2d(out_c))
    layers += [nn.ReLU(inplace=True), nn.MaxPool2d(2)]
    return nn.Sequential(*layers)


class GalaxyCNN(nn.Module):
    """CNN maison : 4 blocs conv + tête dense.

    channels=(32,64,128,256). AdaptiveAvgPool(4x4) fixe la tête quelle que soit
    la résolution d'entrée (64 ou 128 px). ~1,44 M paramètres, dominés par la
    première couche dense (4096 -> 256).

    use_bn et le dropout servent aux ablations.
    """

    def __init__(
        self,
        in_channels: int = 3,
        channels=(32, 64, 128, 256),
        num_classes: int = NUM_CLASSES,
        dropout: float = 0.5,
        use_bn: bool = True,
    ):
        super().__init__()
        blocks: list[nn.Module] = []
        prev = in_channels
        for c in channels:
            blocks.append(_conv_block(prev, c, use_bn=use_bn))
            prev = c
        self.features = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        flat = prev * 4 * 4
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(flat, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )
        # Nom de la dernière conv, utile pour Grad-CAM.
        self.last_conv = self.features[-1][0]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


def build_resnet18(pretrained: bool = True, mode: str = "finetune", num_classes: int = NUM_CLASSES):
    """ResNet18 avec dernière couche remplacée par `num_classes` sorties.

    mode='features'  : backbone gelé, on n'entraîne que fc (feature extraction).
    mode='finetune'  : tout est entraînable.
    """
    from torchvision import models

    weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    net = models.resnet18(weights=weights)
    if mode == "features":
        for p in net.parameters():
            p.requires_grad = False
    net.fc = nn.Linear(net.fc.in_features, num_classes)
    return net


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

### Données — chargement, split, normalisation, augmentation

In [ ]:
"""Chargement, split, normalisation et augmentation de Galaxy10 DECaLS.

Pipeline :
  - décodage HF -> tenseurs uint8 [N,3,res,res] mis en cache par résolution ;
  - split validation stratifié 15 % pris dans le train (le test officiel reste intact) ;
  - normalisation par mean/std calculés sur le train uniquement ;
  - augmentation (rotation 0-360, flips H/V, léger jitter) sur le train seulement.
"""

import numpy as np
import torch
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2



# --- Décodage HF -> tenseurs uint8 ------------------------------------------
def _decode(examples_iter, res: int, n: int | None = None):
    xs, ys = [], []
    for i, ex in enumerate(examples_iter):
        if n is not None and i >= n:
            break
        img: Image.Image = ex["image"].convert("RGB").resize((res, res), Image.BILINEAR)
        xs.append(torch.from_numpy(np.array(img, dtype=np.uint8)).permute(2, 0, 1).contiguous())
        ys.append(int(ex["label"]))
    x = torch.stack(xs)  # uint8 [N,3,res,res]
    y = torch.tensor(ys, dtype=torch.long)
    return x, y


def load_split(split: str, res: int, cache: bool = True):
    """Charge un split complet (train/test) en tenseurs uint8, avec cache disque."""
    from datasets import load_dataset

    cache_path = data_dir() / "cache" / f"{split}_{res}.pt"
    if cache and cache_path.exists():
        blob = torch.load(cache_path)
        return blob["x"], blob["y"]
    ds = load_dataset(HF_DATASET, split=split)
    x, y = _decode(ds, res)
    if cache:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({"x": x, "y": y}, cache_path)
    return x, y


def load_stream(split: str, res: int, n: int):
    """Charge n exemples par streaming (smoke test, sans télécharger tout le parquet)."""
    from datasets import load_dataset

    ds = load_dataset(HF_DATASET, split=split, streaming=True)
    return _decode(ds, res, n=n)


# --- Normalisation ----------------------------------------------------------
def compute_mean_std(x_uint8: torch.Tensor):
    """mean/std par canal sur des images uint8 (renvoie des floats en [0,1])."""
    x = x_uint8.float().div_(255.0)
    mean = x.mean(dim=(0, 2, 3))
    std = x.std(dim=(0, 2, 3))
    return mean, std


# --- Transforms -------------------------------------------------------------
def build_transforms(mean, std, train: bool):
    """train=True -> augmentation physique (rotations/flips). Sinon normalisation seule."""
    mean = tuple(float(m) for m in mean)
    std = tuple(float(s) for s in std)
    if train:
        return v2.Compose([
            v2.RandomHorizontalFlip(),
            v2.RandomVerticalFlip(),
            v2.RandomRotation(180),  # +/-180 deg : une galaxie n'a pas d'orientation privilégiée
            v2.ColorJitter(brightness=0.1, contrast=0.1),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean, std),
        ])
    return v2.Compose([
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean, std),
    ])


class GalaxyDataset(Dataset):
    """Tenseur uint8 en mémoire + transform appliquée à la volée."""

    def __init__(self, x_uint8: torch.Tensor, y: torch.Tensor, transform):
        self.x = x_uint8
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.transform(self.x[i]), self.y[i]


# --- Assemblage -------------------------------------------------------------
def get_datasets(
    res: int = 64,
    val_frac: float = 0.15,
    seed: int = SEED,
    smoke_n: int | None = None,
    cache: bool = True,
    mean=None,
    std=None,
    augment: bool = True,
):
    """Retourne (train_ds, val_ds, test_ds, (mean, std)).

    smoke_n : si vrai (booléen ou entier), charge le petit split test officiel (1800
    images, toutes les classes) et le découpe 60/20/20, uniquement pour vérifier que
    le pipeline tourne sans télécharger les 2,3 Go du train. L'expérience réelle
    utilise smoke_n=None : le test HF reste le test final, la validation vient du train.
    mean/std : passer les stats ImageNet pour le ResNet pré-entraîné ; None = stats train.
    augment=False désactive l'augmentation (ablation).
    """
    if smoke_n is not None:
        x, y = load_split("test", res, cache=cache)  # 1800 imgs, 10 classes garanties
        if isinstance(smoke_n, int) and smoke_n < len(y):
            keep, _ = train_test_split(
                np.arange(len(y)), train_size=smoke_n, stratify=y.numpy(), random_state=seed
            )
            x, y = x[keep], y[keep]
        idx = np.arange(len(y))
        tr, tmp = train_test_split(idx, test_size=0.4, stratify=y.numpy(), random_state=seed)
        val, te = train_test_split(tmp, test_size=0.5, stratify=y.numpy()[tmp], random_state=seed)
        xtr, ytr, xval, yval, xte, yte = x[tr], y[tr], x[val], y[val], x[te], y[te]
    else:
        xall, yall = load_split("train", res, cache=cache)
        xte, yte = load_split("test", res, cache=cache)
        tr, val = train_test_split(
            np.arange(len(yall)), test_size=val_frac, stratify=yall.numpy(), random_state=seed
        )
        xtr, ytr, xval, yval = xall[tr], yall[tr], xall[val], yall[val]

    if mean is None or std is None:
        mean, std = compute_mean_std(xtr)

    train_ds = GalaxyDataset(xtr, ytr, build_transforms(mean, std, train=augment))
    val_ds = GalaxyDataset(xval, yval, build_transforms(mean, std, train=False))
    test_ds = GalaxyDataset(xte, yte, build_transforms(mean, std, train=False))
    return train_ds, val_ds, test_ds, (mean, std)


def make_loaders(train_ds, val_ds, test_ds, batch_size: int = 128, num_workers: int = 0):
    kw = dict(batch_size=batch_size, num_workers=num_workers, pin_memory=False)
    return (
        DataLoader(train_ds, shuffle=True, drop_last=False, **kw),
        DataLoader(val_ds, shuffle=False, **kw),
        DataLoader(test_ds, shuffle=False, **kw),
    )

### Entraînement — boucle, évaluation, early stopping sur F1 macro

In [ ]:
"""Boucle d'entraînement, évaluation, et fit avec early stopping sur F1 macro."""

import copy
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader



def class_weights(labels, num_classes: int = NUM_CLASSES) -> torch.Tensor:
    """Poids = inverse de la fréquence, normalisés. Pour CrossEntropyLoss(weight=)."""
    labels = np.asarray(labels)
    counts = np.bincount(labels, minlength=num_classes).astype(np.float64)
    counts[counts == 0] = 1.0
    w = counts.sum() / (num_classes * counts)
    return torch.tensor(w, dtype=torch.float32)


@torch.no_grad()
def evaluate(model, loader: DataLoader, criterion, device=None):
    """Retourne (loss moyenne, y_true, y_pred). Met le modèle en eval()."""
    device = device or get_device()
    model.eval()
    total_loss, n = 0.0, 0
    ys, ps = [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        loss = criterion(out, yb)
        total_loss += loss.item() * xb.size(0)
        n += xb.size(0)
        ps.append(out.argmax(1).cpu())
        ys.append(yb.cpu())
    y_true = torch.cat(ys).numpy()
    y_pred = torch.cat(ps).numpy()
    return total_loss / n, y_true, y_pred


def train_one_epoch(model, loader, optimizer, criterion, device=None) -> float:
    device = device or get_device()
    model.train()
    total_loss, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        n += xb.size(0)
    return total_loss / n


def fit(
    model,
    train_loader,
    val_loader,
    epochs: int = 30,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    criterion=None,
    optimizer=None,
    scheduler: str | None = "cosine",
    patience: int = 8,
    device=None,
    verbose: bool = True,
):
    """Entraîne et garde les meilleurs poids selon le F1 macro de validation.

    Retourne (best_model, history) où history est un dict de listes par epoch.
    """
    device = device or get_device()
    model.to(device)
    criterion = criterion or nn.CrossEntropyLoss()
    optimizer = optimizer or torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=weight_decay
    )
    sched = None
    if scheduler == "cosine":
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {"train_loss": [], "val_loss": [], "val_f1_macro": [], "val_acc": []}
    best_f1, best_state, best_epoch, since = -1.0, None, -1, 0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, y_true, y_pred = evaluate(model, val_loader, criterion, device)
        f1 = macro_f1(y_true, y_pred)
        acc = float((y_true == y_pred).mean())
        if sched is not None:
            sched.step()

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(val_loss)
        history["val_f1_macro"].append(f1)
        history["val_acc"].append(acc)

        if f1 > best_f1:
            best_f1, best_state, best_epoch, since = f1, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            since += 1

        if verbose:
            print(
                f"epoch {epoch:2d} | train {tr_loss:.3f} | val {val_loss:.3f} "
                f"| val F1 {f1:.3f} | val acc {acc:.3f} | {time.time()-t0:.1f}s"
            )
        if since >= patience:
            if verbose:
                print(f"early stopping (best F1 {best_f1:.3f} @ epoch {best_epoch})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    history["best_epoch"] = best_epoch
    history["best_val_f1_macro"] = best_f1
    return model, history

### Figures

In [ ]:
"""Figures du projet. Chaque fonction sauvegarde dans figures/ et renvoie le chemin."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch



def _save(fig, name: str) -> Path:
    path = figures_dir() / name
    fig.savefig(path, dpi=130, bbox_inches="tight")
    plt.close(fig)
    return path


def _to_hwc(img_uint8: torch.Tensor) -> np.ndarray:
    return img_uint8.permute(1, 2, 0).cpu().numpy().astype(np.uint8)


def plot_samples_grid(x_uint8, y, class_names=CLASS_NAMES, name="samples.png"):
    """Une image par classe (première rencontrée)."""
    y = y.numpy() if torch.is_tensor(y) else np.asarray(y)
    fig, axes = plt.subplots(2, 5, figsize=(13, 5.6))
    for c, ax in enumerate(axes.ravel()):
        idx = np.where(y == c)[0]
        if len(idx):
            ax.imshow(_to_hwc(x_uint8[idx[0]]))
        ax.set_title(f"{c} · {class_names[c]}", fontsize=9)
        ax.axis("off")
    fig.suptitle("Galaxy10 DECaLS — un exemple par classe", fontsize=12)
    return _save(fig, name)


def plot_class_distribution(y, class_names=CLASS_NAMES, name="distribution.png"):
    y = y.numpy() if torch.is_tensor(y) else np.asarray(y)
    counts = np.bincount(y, minlength=len(class_names))
    order = np.argsort(counts)[::-1]
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.bar(range(len(counts)), counts[order], color="#4a86e8")
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels([class_names[i] for i in order], rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("nombre d'images")
    ax.set_title(f"Distribution des classes (déséquilibre, facteur {counts.max()/counts.min():.1f})")
    return _save(fig, name)


def plot_augmentation(img_uint8, mean, std, n=6, name="augment.png"):
    """Une même galaxie augmentée n fois (rotations/flips/jitter), pour illustrer."""
    from torchvision.transforms import v2

    disp = v2.Compose([
        v2.RandomHorizontalFlip(),
        v2.RandomVerticalFlip(),
        v2.RandomRotation(180),
        v2.ColorJitter(brightness=0.1, contrast=0.1),
        v2.ToDtype(torch.float32, scale=True),
    ])
    fig, axes = plt.subplots(1, n + 1, figsize=(2.0 * (n + 1), 2.2))
    axes[0].imshow(_to_hwc(img_uint8))
    axes[0].set_title("original", fontsize=9)
    axes[0].axis("off")
    for k in range(1, n + 1):
        axes[k].imshow(disp(img_uint8).permute(1, 2, 0).clamp(0, 1).numpy())
        axes[k].set_title(f"aug {k}", fontsize=9)
        axes[k].axis("off")
    fig.suptitle("Augmentation (train uniquement) : invariances physiques", fontsize=11)
    return _save(fig, name)


def plot_curves(history, name="courbes.png"):
    ep = range(1, len(history["train_loss"]) + 1)
    fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4.2))
    a.plot(ep, history["train_loss"], label="train")
    a.plot(ep, history["val_loss"], label="val")
    a.set_xlabel("epoch"); a.set_ylabel("loss"); a.set_title("Loss"); a.legend()
    b.plot(ep, history["val_f1_macro"], label="F1 macro", color="#16a765")
    b.plot(ep, history["val_acc"], label="accuracy", color="#e07798")
    b.set_xlabel("epoch"); b.set_ylabel("val"); b.set_title("Validation"); b.legend()
    if history.get("best_epoch", -1) > 0:
        for ax in (a, b):
            ax.axvline(history["best_epoch"], ls="--", c="gray", lw=1)
    return _save(fig, name)


def plot_confusion(cm, class_names=CLASS_NAMES, name="confusion.png", normalized=True):
    fig, ax = plt.subplots(figsize=(8.5, 7.5))
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.max() if cm.max() > 0 else 1)
    ax.set_xticks(range(len(class_names))); ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(class_names, fontsize=8)
    ax.set_xlabel("prédit"); ax.set_ylabel("vrai")
    fmt = ".2f" if normalized else "d"
    thr = cm.max() / 2 if cm.max() > 0 else 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt), ha="center", va="center",
                    color="white" if cm[i, j] > thr else "black", fontsize=7)
    ax.set_title("Matrice de confusion" + (" (normalisée)" if normalized else ""))
    fig.colorbar(im, ax=ax, fraction=0.046)
    return _save(fig, name)


def plot_perclass_f1(per_class, name="perclass.png"):
    """per_class : DataFrame index=classe, colonne 'f1'."""
    f1 = per_class["f1"].values
    names = list(per_class.index)
    order = np.argsort(f1)
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.barh(range(len(f1)), f1[order], color="#43d692")
    ax.set_yticks(range(len(f1)))
    ax.set_yticklabels([names[i] for i in order], fontsize=8)
    ax.set_xlabel("F1"); ax.set_xlim(0, 1)
    ax.set_title("F1 par classe (test)")
    return _save(fig, name)


def plot_scale_comparison(results: dict, metric="f1_macro", name="echelle.png"):
    """results : {nom_modele: {metric: valeur}} -> barres baseline -> CNN -> ResNet."""
    names = list(results.keys())
    vals = [results[k][metric] for k in names]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    colors = ["#cccccc", "#999999", "#4a86e8", "#16a765"][: len(names)]
    ax.bar(names, vals, color=colors)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
    ax.set_ylabel(metric); ax.set_ylim(0, 1)
    ax.set_title("Que gagne-t-on avec la convolution ? (test)")
    return _save(fig, name)

### Grad-CAM

In [ ]:
"""Grad-CAM à la main (Selvaraju et al., 2017).

Principe : pour la classe cible, on prend le gradient du score par rapport aux
feature maps de la dernière conv, on en fait une moyenne spatiale (importance de
chaque canal), on pondère les feature maps, ReLU, puis on redimensionne à la taille
de l'image. La carte montre *où* le réseau regarde.
"""

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F



class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._forward_hook)
        target_layer.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, inp, out):
        self.activations = out.detach()

    def _backward_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __call__(self, x, class_idx: int | None = None):
        """x : [1,3,H,W] normalisé. Retourne (heatmap [H,W] en [0,1], classe)."""
        self.model.eval()
        out = self.model(x)
        if class_idx is None:
            class_idx = int(out.argmax(1).item())
        self.model.zero_grad()
        out[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)      # [1,C,1,1]
        cam = F.relu((weights * self.activations).sum(1, keepdim=True))  # [1,1,h,w]
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.cpu().numpy(), class_idx


def _overlay(img_hwc_uint8: np.ndarray, cam: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    heat = plt.get_cmap("jet")(cam)[..., :3]  # [H,W,3] float
    base = img_hwc_uint8.astype(np.float32) / 255.0
    return np.clip((1 - alpha) * base + alpha * heat, 0, 1)


def plot_gradcam_grid(
    model,
    target_layer,
    images_uint8,
    mean,
    std,
    y_true=None,
    class_names=CLASS_NAMES,
    device=None,
    name="gradcam.png",
):
    """Pour chaque image (uint8 [3,H,W]) : original + carte Grad-CAM superposée."""
    device = device or get_device()
    model.to(device)
    cam = GradCAM(model, target_layer)
    tfm = build_transforms(mean, std, train=False)

    m = len(images_uint8)
    fig, axes = plt.subplots(2, m, figsize=(2.4 * m, 5.0))
    if m == 1:
        axes = axes.reshape(2, 1)
    for k in range(m):
        img = images_uint8[k]
        x = tfm(img).unsqueeze(0).to(device)
        heat, pred = cam(x)
        hwc = img.permute(1, 2, 0).cpu().numpy().astype(np.uint8)
        axes[0, k].imshow(hwc)
        title = class_names[pred]
        if y_true is not None:
            t = int(y_true[k])
            ok = "OK" if t == pred else f"vrai {class_names[t]}"
            title = f"pred {class_names[pred]}\n{ok}"
        axes[0, k].set_title(title, fontsize=8)
        axes[0, k].axis("off")
        axes[1, k].imshow(_overlay(hwc, heat))
        axes[1, k].axis("off")
    fig.suptitle("Grad-CAM : où le réseau regarde", fontsize=12)
    path = figures_dir() / name
    fig.savefig(path, dpi=130, bbox_inches="tight")
    plt.close(fig)
    return path

## 1. Analyse exploratoire (EDA)

On charge les données et on regarde ce qu'on manipule : un exemple par classe, la
distribution (fortement déséquilibrée), et les statistiques de couleur par classe.

In [ ]:
set_seed()
device = get_device(); print("device:", device)

train_ds, val_ds, test_ds, (mean, std) = get_datasets(res=RES_SMALL, smoke_n=SMOKE_N)
print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")
display(ShowImage(plot_samples_grid(test_ds.x, test_ds.y)))

In [ ]:
# Distribution des classes sur l'ensemble d'entraînement (train + val)
y_full = torch.cat([train_ds.y, val_ds.y])
display(ShowImage(plot_class_distribution(y_full)))

In [ ]:
# Statistiques de couleur moyenne par classe (sur le train, en [0,1])
xtr = train_ds.x.float() / 255.0
rows = []
for c in range(NUM_CLASSES):
    m = xtr[train_ds.y == c].mean(dim=(0, 2, 3))
    rows.append([CLASS_NAMES[c], int((train_ds.y == c).sum()), *[round(v, 3) for v in m.tolist()]])
pd.DataFrame(rows, columns=["classe", "n_train", "R", "G", "B"])

## 2. Preprocessing et augmentation

**Split.** Le dataset fournit déjà un `train` (16 000) et un `test` (1 800). On garde le
**test officiel comme test final, jamais touché**. La validation est un tirage **stratifié
15 %** dans le train (seed figé). Toutes les décisions de tuning se prennent sur la validation.

**Normalisation.** mean/std par canal calculés **sur le train uniquement**, pour éviter toute
fuite d'information du test.

**Augmentation (train uniquement).** Rotations 0-360°, flips horizontaux et verticaux, léger
jitter de luminosité/contraste. Justification physique : une galaxie n'a pas d'orientation
privilégiée dans le ciel, ces transformations sont des invariances réelles du problème. On les
applique **après le split**, jamais sur val/test.

In [ ]:
print("mean train:", [round(v, 4) for v in mean.tolist()])
print("std  train:", [round(v, 4) for v in std.tolist()])
display(ShowImage(plot_augmentation(train_ds.x[0], mean, std)))

## 3. Baselines sans convolution

Elles doivent être médiocres : c'est le but. Elles aplatissent l'image et traitent chaque
pixel comme une variable indépendante, sans aucune notion de structure spatiale. On pondère la
loss par l'inverse des fréquences (déséquilibre) et on suit le **F1 macro**.

In [ ]:
tl, vl, te = make_loaders(train_ds, val_ds, test_ds, batch_size=BATCH, num_workers=NUM_WORKERS)
cw = class_weights(train_ds.y).to(device)
crit = nn.CrossEntropyLoss(weight=cw)
results = {}

logreg = LogisticRegression(image_size=RES_SMALL)
logreg, _ = fit(logreg, tl, vl, epochs=EPOCHS_BASE, lr=1e-3, criterion=crit,
                scheduler=None, device=device, verbose=False)
_, yt, yp = evaluate(logreg, te, crit, device)
results["LogReg"] = summary_metrics(yt, yp)

mlp = MLP(image_size=RES_SMALL)
mlp, _ = fit(mlp, tl, vl, epochs=EPOCHS_BASE, lr=1e-3, criterion=crit,
             scheduler=None, device=device, verbose=False)
_, yt, yp = evaluate(mlp, te, crit, device)
results["MLP"] = summary_metrics(yt, yp)

pd.DataFrame(results).T[["accuracy", "f1_macro", "f1_weighted"]].round(3)

## 4. CNN maison et tuning

4 blocs `[Conv 3x3 (sans biais) → BatchNorm → ReLU → MaxPool]` avec channels
`(32, 64, 128, 256)`, `AdaptiveAvgPool(4x4)`, puis une tête dense. La conv 3x3 avec padding 1
conserve la taille ; le pooling la divise par 2. Le partage de poids d'un filtre sur toute
l'image donne l'**équivariance par translation** : déplacer un motif dans l'image déplace
d'autant sa réponse.

In [ ]:
cnn = GalaxyCNN()
print("paramètres entraînables:", f"{count_parameters(cnn):,}")

# Dimensions de sortie bloc par bloc (entrée 3 x RES_SMALL x RES_SMALL)
x = torch.zeros(1, 3, RES_SMALL, RES_SMALL); shapes = []
for i, blk in enumerate(cnn.features):
    x = blk(x); shapes.append([f"bloc {i+1}", tuple(x.shape[1:])])
display(pd.DataFrame(shapes, columns=["couche", "sortie (C, H, W)"]))

In [ ]:
cnn, hist = fit(cnn, tl, vl, epochs=EPOCHS_CNN, lr=1e-3, criterion=crit, device=device)
display(ShowImage(plot_curves(hist)))
_, yt, yp = evaluate(cnn, te, crit, device)
results["CNN"] = summary_metrics(yt, yp)
print("CNN test:", {k: round(v, 3) for k, v in results["CNN"].items()})

**Tuning du learning rate** (le plus important). On garde le meilleur F1 macro de validation.

In [ ]:
runs = []
for lr in [3e-4, 1e-3, 3e-3]:
    m = GalaxyCNN()
    m, h = fit(m, tl, vl, epochs=TUNE_EPOCHS, lr=lr, criterion=crit, device=device, verbose=False)
    runs.append({"lr": lr, "val_f1_macro": round(h["best_val_f1_macro"], 3),
                 "best_epoch": h["best_epoch"]})
pd.DataFrame(runs)

## 5. Transfer learning — ResNet18

On réutilise un ResNet18 pré-entraîné sur ImageNet, on remplace la dernière couche par 10
sorties, et on procède en deux temps : d'abord **feature extraction** (backbone gelé, on
n'entraîne que la tête), puis **fine-tuning** (tout est dégelé, LR plus faible). On travaille
en 128 px et on normalise avec les statistiques ImageNet, attendues par le backbone.

In [ ]:
tr2, val2, te2ds, _ = get_datasets(res=RES_LARGE, smoke_n=SMOKE_N,
                                   mean=IMAGENET_MEAN, std=IMAGENET_STD)
tl2, vl2, te2 = make_loaders(tr2, val2, te2ds, batch_size=BATCH, num_workers=NUM_WORKERS)
cw2 = class_weights(tr2.y).to(device)
crit2 = nn.CrossEntropyLoss(weight=cw2)

resnet = build_resnet18(pretrained=True, mode="features")
resnet, _ = fit(resnet, tl2, vl2, epochs=EPOCHS_RES_FE, lr=1e-3, criterion=crit2,
                device=device, verbose=False)                    # tête seule
for p in resnet.parameters():
    p.requires_grad = True
resnet, hist_res = fit(resnet, tl2, vl2, epochs=EPOCHS_RES, lr=1e-4, criterion=crit2,
                       device=device)                            # fine-tuning complet
_, yt2, yp2 = evaluate(resnet, te2, crit2, device)
results["ResNet18"] = summary_metrics(yt2, yp2)
print("ResNet18 test:", {k: round(v, 3) for k, v in results["ResNet18"].items()})

## 6. Évaluation sur le test

L'échelle baseline → CNN → ResNet répond à la question « que gagne-t-on avec la
convolution ? ». On regarde ensuite la matrice de confusion et le F1 par classe du meilleur
modèle, et on analyse les confusions entre classes voisines.

In [ ]:
display(ShowImage(plot_scale_comparison(results, metric="f1_macro")))
display(pd.DataFrame(results).T[["accuracy", "f1_macro", "f1_weighted"]].round(3))

In [ ]:
# Meilleur modèle = ResNet18 : confusion + F1 par classe
display(ShowImage(plot_confusion(confusion(yt2, yp2, normalize="true"))))
pc = per_class_df(yt2, yp2)
display(ShowImage(plot_perclass_f1(pc)))
pc.round(3)

In [ ]:
# Sauvegarde des résultats pour le rapport (injection des chiffres)
results_out = {
    "models": {k: {kk: round(vv, 4) for kk, vv in v.items()} for k, v in results.items()},
    "cnn_params": count_parameters(GalaxyCNN()),
    "cnn_best_epoch": hist.get("best_epoch"),
    "resnet_best_epoch": hist_res.get("best_epoch"),
    "per_class_f1_resnet": {CLASS_NAMES[i]: round(float(pc["f1"].iloc[i]), 3)
                            for i in range(NUM_CLASSES)},
    "tuning_lr": runs,
    "config": {"res_small": RES_SMALL, "res_large": RES_LARGE, "batch": BATCH,
               "full_run": FULL_RUN},
}
out_path = project_root() / "results.json"
json.dump(results_out, open(out_path, "w"), indent=2, ensure_ascii=False)
print("écrit:", out_path)
print(json.dumps(results_out["models"], indent=2, ensure_ascii=False))

## 7. Grad-CAM — où le réseau regarde

Grad-CAM pondère les feature maps de la dernière convolution par le gradient moyen du score
de la classe, applique une ReLU, et rééchelonne à la taille de l'image. On l'applique sur des
prédictions correctes **et** des erreurs, pour le CNN maison et pour le ResNet : le réseau
regarde-t-il les bras spiraux, la barre, le compagnon de fusion, ou des artefacts ?

In [ ]:
def pick_examples(model, ds, device, n_ok=3, n_bad=3):
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in DataLoader(ds, batch_size=256):
            preds.append(model(xb.to(device)).argmax(1).cpu())
    preds = torch.cat(preds)
    ok = (preds == ds.y).nonzero().flatten().tolist()
    bad = (preds != ds.y).nonzero().flatten().tolist()
    return ok[:n_ok] + bad[:n_bad]

idx = pick_examples(cnn, test_ds, device)
display(ShowImage(plot_gradcam_grid(cnn, cnn.last_conv, test_ds.x[idx], mean, std,
                                    y_true=test_ds.y[idx], device=device, name="gradcam_cnn.png")))

In [ ]:
idx2 = pick_examples(resnet, te2ds, device)
display(ShowImage(plot_gradcam_grid(resnet, resnet.layer4[-1], te2ds.x[idx2],
                                    IMAGENET_MEAN, IMAGENET_STD, y_true=te2ds.y[idx2],
                                    device=device, name="gradcam_resnet.png")))

## 8. Ablations

On retire un ingrédient à la fois pour quantifier son apport : **sans augmentation** et
**sans BatchNorm**. On compare le F1 macro (test) au CNN complet.

In [ ]:
abl = {"CNN complet": results["CNN"]["f1_macro"]}

# sans augmentation
tr_na, val_na, te_na, (m_na, s_na) = get_datasets(res=RES_SMALL, smoke_n=SMOKE_N, augment=False)
tl_na, vl_na, te_na_l = make_loaders(tr_na, val_na, te_na, batch_size=BATCH, num_workers=NUM_WORKERS)
cnn_na, _ = fit(GalaxyCNN(), tl_na, vl_na, epochs=EPOCHS_CNN, lr=1e-3, criterion=crit,
                device=device, verbose=False)
_, yt, yp = evaluate(cnn_na, te_na_l, crit, device)
abl["sans augmentation"] = summary_metrics(yt, yp)["f1_macro"]

# sans BatchNorm
cnn_nobn, _ = fit(GalaxyCNN(use_bn=False), tl, vl, epochs=EPOCHS_CNN, lr=1e-3, criterion=crit,
                  device=device, verbose=False)
_, yt, yp = evaluate(cnn_nobn, te, crit, device)
abl["sans BatchNorm"] = summary_metrics(yt, yp)["f1_macro"]

# On ajoute les ablations à results.json (pour l'injection dans le rapport).
try:
    _r = json.load(open(project_root() / "results.json"))
    _r["ablations"] = {k: round(v, 4) for k, v in abl.items()}
    json.dump(_r, open(project_root() / "results.json", "w"), indent=2, ensure_ascii=False)
except Exception as e:
    print("maj results.json:", e)

pd.DataFrame({"f1_macro (test)": {k: round(v, 3) for k, v in abl.items()}})

## Conclusion

L'échelle baseline → CNN → ResNet montre concrètement l'apport de la convolution, puis du
transfer learning. Grad-CAM confirme que le réseau s'appuie sur les structures morphologiques
pertinentes. Limites : labels bruités de Galaxy Zoo, fort déséquilibre (la classe Cigar Shaped
à 334 images reste la plus difficile), et coût d'entraînement.

**Pour le run complet sur Colab :** activer le GPU T4, garder `FULL_RUN=True`, exécuter tout le
notebook, puis récupérer `figures/` et `results.json`.